# Ariadne Index Types

This notebook demonstrates all six index types available in Ariadne.
Each section creates an index, adds the appropriate index type, and shows how it works.

In [1]:
%%init_spark
launcher.master = "local[*]"
launcher.conf.set("spark.ariadne.storagePath", "/home/ariadne_storage")
launcher.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
launcher.conf.set("spark.sql.catalog.ariadne", "dev.cjfravel.ariadne.catalog.AriadneCatalog")
launcher.conf.set("spark.sql.extensions", "dev.cjfravel.ariadne.catalog.AriadneSparkExtension")

In [2]:
import dev.cjfravel.ariadne.Index
import org.apache.spark.sql.types._
implicit val spark = org.apache.spark.sql.SparkSession.builder().getOrCreate()

// Clean up any previous index data
import org.apache.hadoop.fs.{FileSystem, Path}
val fs = FileSystem.get(spark.sparkContext.hadoopConfiguration)
fs.delete(new Path("/home/ariadne_storage"), true)


Intitializing Scala interpreter ...

Spark Web UI available at http://a1c023bd9c3e:4040
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1775581940244)
SparkSession available as 'spark'

import dev.cjfravel.ariadne.Index
import org.apache.spark.sql.types._
spark: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@1fa9a354
import org.apache.hadoop.fs.{FileSystem, Path}
fs: org.apache.hadoop.fs.FileSystem = org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem@3f4e4d63
res0: Boolean = false

## 1. Regular Index

The most common index type. Stores distinct values per file in an array column.
Best for columns with moderate cardinality that you join or filter on frequently.

**Use case:** Customer ID lookups — quickly find which files contain a given customer.

In [3]:
val customerSchema = StructType(Seq(
  StructField("customer_id", IntegerType),
  StructField("name", StringType),
  StructField("email", StringType),
  StructField("region", StringType),
  StructField("signup_date", StringType)
))

val customers = Index("customers", customerSchema, "csv", Map("header" -> "true"))
customers.addFile("/home/data/customers_west.csv")
customers.addFile("/home/data/customers_east.csv")

// Add a regular index on customer_id
customers.addIndex("customer_id")
customers.update

customerSchema: org.apache.spark.sql.types.StructType = StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))
customers: dev.cjfravel.ariadne.Index = Index(customers,Some(StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))))

In [4]:
// locateFiles returns which source files contain customer 1001
val files = customers.locateFiles(Map("customer_id" -> Array(1001)))
println(s"Files containing customer 1001: ${files.mkString(", ")}")

// Query for a customer in the East file
val eastFiles = customers.locateFiles(Map("customer_id" -> Array(2001)))
println(s"Files containing customer 2001: ${eastFiles.mkString(", ")}")

Files containing customer 1001: file:///home/data/customers_west.csv
Files containing customer 2001: file:///home/data/customers_east.csv


files: Set[String] = Set(file:///home/data/customers_west.csv)
eastFiles: Set[String] = Set(file:///home/data/customers_east.csv)

## 2. Bloom Filter Index

A probabilistic index that uses much less storage than a regular index.
Returns *possible* matches (may include false positives but never false negatives).

**Use case:** High-cardinality columns like product SKUs where storing every distinct
value per file would be too expensive.

In [5]:
val orderSchema = StructType(Seq(
  StructField("order_id", IntegerType),
  StructField("customer_id", IntegerType),
  StructField("product_sku", StringType),
  StructField("amount", DoubleType),
  StructField("order_date", StringType)
))

val orders = Index("orders", orderSchema, "csv", Map("header" -> "true"))
orders.addFile("/home/data/orders_q1.csv")
orders.addFile("/home/data/orders_q2.csv")

// Add a bloom filter index on product_sku (1% false positive rate)
orders.addBloomIndex("product_sku", 0.01)
orders.addIndex("customer_id")
orders.update

orderSchema: org.apache.spark.sql.types.StructType = StructType(StructField(order_id,IntegerType,true),StructField(customer_id,IntegerType,true),StructField(product_sku,StringType,true),StructField(amount,DoubleType,true),StructField(order_date,StringType,true))
orders: dev.cjfravel.ariadne.Index = Index(orders,Some(StructType(StructField(order_id,IntegerType,true),StructField(customer_id,IntegerType,true),StructField(product_sku,StringType,true),StructField(amount,DoubleType,true),StructField(order_date,StringType,true))))

In [6]:
// Find files that might contain SKU-D400
val skuFiles = orders.locateFiles(Map("product_sku" -> Array("SKU-D400")))
println(s"Files possibly containing SKU-D400: ${skuFiles.mkString(", ")}")

// SKU-D400 only appears in orders_q2.csv — bloom filter should narrow it down

Files possibly containing SKU-D400: file:///home/data/orders_q2.csv


skuFiles: Set[String] = Set(file:///home/data/orders_q2.csv)

## 3. Computed Index

Indexes a column derived from a Spark SQL expression.
The computed column is created at index time and available for queries.

**Use case:** Index the email domain extracted from an email address,
so you can quickly find all customers from a given company.

In [7]:
val customersComputed = Index("customers_computed", customerSchema, "csv", Map("header" -> "true"))
customersComputed.addFile("/home/data/customers_west.csv")
customersComputed.addFile("/home/data/customers_east.csv")

// Create a computed index on the email domain
customersComputed.addComputedIndex("email_domain", "split(email, '@')[1]")
customersComputed.update

customersComputed: dev.cjfravel.ariadne.Index = Index(customers_computed,Some(StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))))

In [8]:
// Find files containing customers with @bigcorp.com emails
val corpFiles = customersComputed.locateFiles(Map("email_domain" -> Array("bigcorp.com")))
println(s"Files with bigcorp.com customers: ${corpFiles.mkString(", ")}")

// Both files have bigcorp.com customers (Grace in West, Frank in East)

Files with bigcorp.com customers: file:///home/data/customers_east.csv, file:///home/data/customers_west.csv


corpFiles: Set[String] = Set(file:///home/data/customers_east.csv, file:///home/data/customers_west.csv)

## 4. Temporal Index

For time-series data where each entity has multiple versions across files.
The temporal index tracks which file contains the *latest* version of each value,
and joins automatically deduplicate to return only the most recent version.

**Use case:** IoT sensor readings — when joining, you only want the latest
reading for each sensor, not historical values.

In [9]:
val sensorSchema = StructType(Seq(
  StructField("sensor_id", StringType),
  StructField("temperature", DoubleType),
  StructField("humidity", DoubleType),
  StructField("reading_time", TimestampType)
))

val sensors = Index("sensor_readings", sensorSchema, "csv", Map("header" -> "true"))
sensors.addFile("/home/data/sensor_readings_jan.csv")
sensors.addFile("/home/data/sensor_readings_feb.csv")

// Temporal index: sensor_id versioned by reading_time
sensors.addTemporalIndex("sensor_id", "reading_time")
sensors.update

sensorSchema: org.apache.spark.sql.types.StructType = StructType(StructField(sensor_id,StringType,true),StructField(temperature,DoubleType,true),StructField(humidity,DoubleType,true),StructField(reading_time,TimestampType,true))
sensors: dev.cjfravel.ariadne.Index = Index(sensor_readings,Some(StructType(StructField(sensor_id,StringType,true),StructField(temperature,DoubleType,true),StructField(humidity,DoubleType,true),StructField(reading_time,TimestampType,true))))

In [10]:
// locateFiles with temporal index returns only the file with the LATEST version
val s001Files = sensors.locateFiles(Map("sensor_id" -> Array("S001")))
println(s"Latest file for S001: ${s001Files.mkString(", ")}")
// S001 appears in both files, but Feb has the latest reading

// Joining automatically deduplicates to latest version
val queryDf = spark.createDataFrame(
  spark.sparkContext.parallelize(Seq(org.apache.spark.sql.Row("S001"), org.apache.spark.sql.Row("S002"))),
  StructType(Seq(StructField("sensor_id", StringType)))
)
val latest = sensors.join(queryDf, Seq("sensor_id"), "inner")
latest.show()
// Returns only the Feb readings (latest) for S001 and S002


Latest file for S001: file:///home/data/sensor_readings_feb.csv
+---------+-----------+--------+-------------------+
|sensor_id|temperature|humidity|       reading_time|
+---------+-----------+--------+-------------------+
|     S001|       22.0|    46.0|2024-02-10 12:00:00|
|     S002|       17.5|    65.0|2024-02-10 08:00:00|
+---------+-----------+--------+-------------------+



s001Files: Set[String] = Set(file:///home/data/sensor_readings_feb.csv)
queryDf: org.apache.spark.sql.DataFrame = [sensor_id: string]
latest: org.apache.spark.sql.DataFrame = [sensor_id: string, temperature: double ... 2 more fields]

## 5. Range Index

Stores the min/max values per file for a column. Efficient for range queries
where you need to find files that *could* contain values in a range.

**Use case:** Find orders within a price range without scanning all files.

In [11]:
val ordersRange = Index("orders_range", orderSchema, "csv", Map("header" -> "true"))
ordersRange.addFile("/home/data/orders_q1.csv")
ordersRange.addFile("/home/data/orders_q2.csv")

// Range index on order amount
ordersRange.addRangeIndex("amount")
ordersRange.update

ordersRange: dev.cjfravel.ariadne.Index = Index(orders_range,Some(StructType(StructField(order_id,IntegerType,true),StructField(customer_id,IntegerType,true),StructField(product_sku,StringType,true),StructField(amount,DoubleType,true),StructField(order_date,StringType,true))))

In [12]:
// Find files containing orders around $199.99
val highValueFiles = ordersRange.locateFiles(Map("amount" -> Array(199.99)))
println(s"Files with amount=199.99: ${highValueFiles.mkString(", ")}")
// Q1 range is [29.99, 149.99], Q2 range is [29.99, 199.99]
// Only Q2 file matches

Files with amount=199.99: file:///home/data/orders_q2.csv


highValueFiles: Set[String] = Set(file:///home/data/orders_q2.csv)

## 6. Exploded Field Index

Indexes values from nested array fields in JSON/Parquet data.
Extracts a field path from each array element and indexes the flattened values.


**Use case:** Find events by tag name when tags are stored as a nested array
of structs like `tags: [{name: "homepage", category: "navigation"}, ...]`.

In [13]:
// Read JSON to infer schema
val eventsSchema = spark.read.json("/home/data/events.json").schema
println("Events schema:")
eventsSchema.printTreeString()

val events = Index("events", eventsSchema, "json")
events.addFile("/home/data/events.json")

// Index the "name" field from each element in the "tags" array
events.addExplodedFieldIndex("tags", "name", "tag_name")
events.addIndex("event_type")
events.update

Events schema:
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_ids: array (nullable = true)
 |    |-- element: long (containsNull = true)



eventsSchema: org.apache.spark.sql.types.StructType = StructType(StructField(event_id,StringType,true),StructField(event_type,StringType,true),StructField(tags,ArrayType(StructType(StructField(category,StringType,true),StructField(name,StringType,true)),true),true),StructField(timestamp,StringType,true),StructField(user_ids,ArrayType(LongType,true),true))
events: dev.cjfravel.ariadne.Index = Index(events,Some(StructType(StructField(event_id,StringType,true),StructField(event_type,StringType,true),StructField(tags,ArrayType(StructType(StructField(category,StringType,true),StructField(name,StringType,true)),true),true),StructField(timestamp,StringType,true),StructField(user_ids,ArrayType(LongType,true),true))))

In [14]:
// The exploded field index enables file lookups by nested array values
// View the index statistics
events.stats().show(false)


+----------+---------+---------+---------+---------+------------+------+
|Column    |FileCount|MinValues|MaxValues|AvgValues|MedianValues|StdDev|
+----------+---------+---------+---------+---------+------------+------+
|event_type|1        |3        |3        |3.0      |3           |NULL  |
|tags      |1        |6        |6        |6.0      |6           |NULL  |
+----------+---------+---------+---------+---------+------------+------+



## Summary

| Index Type | Method | Best For |
|-----------|--------|----------|
| Regular | `addIndex(col)` | Moderate cardinality equality lookups |
| Bloom | `addBloomIndex(col, fpr)` | High cardinality with acceptable false positives |
| Computed | `addComputedIndex(name, expr)` | Derived values (domain from email, year from date) |
| Temporal | `addTemporalIndex(col, tsCol)` | Time-series with latest-version semantics |
| Range | `addRangeIndex(col)` | Min/max range pruning |
| Exploded Field | `addExplodedFieldIndex(arr, path, as)` | Nested array fields in JSON/Parquet |

Each column can have exactly **one** index type. Attempting to add a second type throws an error.